# PerovStats demo notebook
### This notebook is for demonstrating the processes involved in PerovStats, from deciding on configuration options and running the modules.

If you have any issues please email t.allwood@sheffield.ac.uk - I will get back to you as soon as I can

#### Before running this notebook please read the installation instructions (in the `/docs/` folder) to make sure the project dependencies are set up for the program.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
import copy

from loguru import logger
from pathlib import Path
import yaml
from topostats.io import LoadScans
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sys.path.append(os.path.abspath(os.path.join('..')))
from src.perovstats.core.classes import PerovStats, ImageData
from src.perovstats.core.io import save_to_csv, save_config
from src.perovstats.core.image_processing import normalise_array
from src.perovstats.cli import setup_logger
from src.perovstats.fourier import split_frequencies
from src.perovstats.segmentation import segment_image
from src.perovstats.smears import find_smear_areas
from src.perovstats.processing import format_time
from src.perovstats.grains import find_grains

---
## Setup

#### These options can be changed to suit your needs. the `output_dir` folder will contain a selection of images from various stages in the process as well as the `.csv` files generated.
In the full version of the notebook, `img_file` will be replaced with `img_files` and will need to point to a directory containing .spm files rather than a single specific file.
There are two options for files to test this program on. The first example contains smear areas and the second does not.

You are of course able to test with your own AFM files by replacing the path in the double quotes in `img_file` with your own filepath.

In [ ]:
img_file = [Path("../example images/4_c60_perovonsil_ref_10um.PFQNM.spm")]
# img_file = [Path("../example images/TR_5um_ST4_14_26_per_Si_tapp4.0_00010.spm")]
output_dir = Path("./output")
config_path = Path("../src/perovstats/default_config.yaml")

#### There are two ways of segmenting perovskite grains depending on whether you want to prioritise speed or accuracy
- `traditional`: As the name suggests, this option uses traditional, hard-coded, segmentation methods to outline grain borders. This method is fairly quick (only taking a few seconds per image) but can come with inaccuracies throughout the mask and is prone to big issues if config parameters are not set correctly.
- `cellpose`: Cellpose is a machine learning model used for grain segmentation in images similar to a perovskite scan. It has a much higher accuracy than traditional methods and does not require any configuration options to be entered. This takes much longer (a few minutes) but running PerovStats on a computer with an Nvidia GPU will greatly decrease this time.

For the demonstration purposes of this notebook both methods will be run to show the differences in speed and accuracy between the two. In the full run in the other notebook you will have a space to select the method you'd like for a run.

##### `min_rms` is the variable currently used to adjust the frequency chosen when performing a fourier transform (separation of silicon and perovskite bumps)

- The default value is 10 which has been working well for images used in testing
- If silicon 'mountains' are still visible in the high-passed image lower the `min_rms` value
- If the perovskite grains don't look defined enough increase the `min_rms` value
- This does not have to be a full number and it is recommended to only adjust the value by a maximum of 2 inbetween runs, I don't expect you'll ever want a value under ~7 or over ~15

##### If traditional segmentation has been selected a sensitivity parameter will need to be configured for the segmentation.
    
- **If cellpose segmentation has been selected you can ignore this variable.**

`threshold_offset`
- This is the amount to shift the threshold chosen by the program by before segmentation.
- Lower values lower the threshold and make the segmentation more sensitive, if it is too low for an image grains may be split into multiple
- Higher values raise the threshold making the segmentation less sensitive, if it's too high then multiple grains may be joined together
- When testing for an ideal offset, change the value by ~0.5-1 (or less for fine tuning)

#### Adjust the values in the code below and then re-run the notebook to see how that changes things:

In [ ]:
min_rms = 10

threshold_offset = -0.5

---
## PerovStats Process

You do not need to change any of the below code, just scroll through as the program runs to see the generated images/ log messages through the process.

#### First, the files are loaded and the original heightmap image is extracted.

In [ ]:
setup_logger()

with config_path.open() as f:
    config = yaml.safe_load(f)

time_start = time.perf_counter()

# Load scans
load_config = config["loading"]
loadscans = LoadScans(img_file, config)
try:
    loadscans.get_data()
except ValueError as e:
    logger.warning(e)
    logger.warning(f"Channel {load_config['channel']} not found in file. Please ensure the config option is correct and all files contain the required channel.")
image_dicts = loadscans.img_dict

config["fourier"]["min_rms"] = min_rms

segmentation_method = "traditional"
config["segmentation"]["segmentation_type"] = segmentation_method
config["segmentation"]["traditional"]["threshold_offset"] = threshold_offset

# Create the dataclasses for the whole process and for each image found
perovstats_object = PerovStats(config=config, images=[])
for filename, topostats_object in image_dicts.items():
    image_data = ImageData(
        success=True,
        filename=filename,
        pixel_to_nm_scaling=topostats_object.pixel_to_nm_scaling,
        image_original=topostats_object.image_original,
        image_flattened=None)
    perovstats_object.images.append(image_data)

logger.info("----------------------------------------------------------")
logger.info(f"Loaded {len(perovstats_object.images)} images")

idx = 0
image_object = perovstats_object.images[0]

plt.imshow(image_object.image_original)
plt.title("Original image")
plt.axis("off")
plt.show()

In [ ]:
logger.info("----------------------------------------------------------")
logger.info(f"Processing {image_object.filename} ({idx+1}/{len(perovstats_object.images)})")
logger.info("----------------------------------------------------------")
logger.debug(f"[{image_object.filename}] : Image dimensions: {image_object.image_original.shape}")
logger.debug(f"[{image_object.filename}] : pixel_to_nm_scaling: {image_object.pixel_to_nm_scaling}")

---
#### A fourier transform is then performed on the image which splits it into two sections, one containing the perovskite topology and one containing the silicon topology (to be discarded):

In [ ]:
# Apply fourier transform to split the image into a low-passed and high-passed image
split_frequencies(perovstats_object.config, image_object)

# Show the split images
fig, ax = plt.subplots(1, 2)

ax[0].imshow(image_object.high_pass)
ax[0].set_title("High passed image (Perovskite)")
ax[0].axis("off")

ax[1].imshow(image_object.low_pass)
ax[1].set_title("Low passed image (Silicon)")
ax[1].axis("off")

plt.tight_layout()
plt.show()

---
#### The high-passed image must now be segmented in order to find individual grain outlines. If `cellpose` has been selected as the segmentation method the program will use a machine learning model and may take a few minutes depending on your device. For the quickest results, use a computer that has an Nvidia GPU:

When cellpose mask segmentation completes for an image warnings may appear. As long as the green `SUCCESS` log appears under it, you can ignore these warnings.

For this demo both methods are used and results will be displayed side by side.

In [ ]:
from IPython.display import display, Markdown

perovstats_object_trad = copy.deepcopy(perovstats_object)
perovstats_object_trad.config["segmentation"]["segmentation_type"] = "traditional"
image_object_trad = copy.deepcopy(perovstats_object_trad.images[0])
perovstats_object_cellpose = copy.deepcopy(perovstats_object)
perovstats_object_cellpose.config["segmentation"]["segmentation_type"] = "cellpose"
image_object_cellpose = copy.deepcopy(perovstats_object_cellpose.images[0])

# Generate grain mask of the high-passed image
trad_start_time = time.perf_counter()
display(Markdown("### Traditional segmentation"))
segment_image(perovstats_object_trad.config, image_object_trad)
trad_end_time = time.perf_counter()
trad_time = format_time(trad_end_time - trad_start_time)

cell_start_time = time.perf_counter()
display(Markdown("### Cellpose ML segmentation"))
segment_image(perovstats_object_cellpose.config, image_object_cellpose)
cell_end_time = time.perf_counter()
cell_time = format_time(cell_end_time - cell_start_time)

# Display the generated masks
fig, ax = plt.subplots(1, 2)

ax[0].imshow(image_object_trad.mask)
ax[0].set_title(f"Traditionally generated mask ({trad_time})")
ax[0].axis("off")

ax[1].imshow(image_object_cellpose.mask)
ax[1].set_title(f"Cellpose generated mask ({cell_time})")
ax[1].axis("off")

plt.tight_layout()
plt.show()

# Display the generated masks overlayed onto the high pass image
fig, ax = plt.subplots(1, 2)

mask_overlay_trad = np.stack((image_object_trad.high_pass,)*3, axis=-1)
mask_overlay_trad = normalise_array(mask_overlay_trad)
mask_overlay_trad[image_object_trad.mask > 0] = [1, 0, 0]

mask_overlay_cellpose = np.stack((image_object_cellpose.high_pass,)*3, axis=-1)
mask_overlay_cellpose = normalise_array(mask_overlay_cellpose)
mask_overlay_cellpose[image_object_cellpose.mask > 0] = [1, 0, 0]

ax[0].imshow(mask_overlay_trad)
ax[0].set_title("Mask overlay traditional")
ax[0].axis("off")

ax[1].imshow(mask_overlay_cellpose)
ax[1].set_title("Mask overlay cellpose")
ax[1].axis("off")

plt.tight_layout()
plt.show()

---
#### Smear areas are errors in the AFM scan from the tip not being able to dip low enough after a peak, resulting in horizontal lines which should not be counted in the data. They are identified and grains within these areas are removed:

In [ ]:
# Find smear areas to be ignored/ removed
find_smear_areas(perovstats_object_trad.config, image_object_trad)
find_smear_areas(perovstats_object_cellpose.config, image_object_cellpose)

---
##### The mask (which is currently a collection of thin lines) should then be analysed in order to identify individual grains.
Once this has been done grains touching either the edge of the image or an identified smear area can be removed.

In [ ]:
# Identify individual grains from mask and generate statistics on them
find_grains(perovstats_object_trad.config, image_object_trad) 
find_grains(perovstats_object_cellpose.config, image_object_cellpose)

Stats can also now be taken from these grains and displayed in graphs:

In [ ]:
highpass_overlay_trad = np.stack((image_object_trad.high_pass,)*3, axis=-1)
highpass_overlay_trad = normalise_array(highpass_overlay_trad)
highpass_overlay_trad[image_object_trad.cleaned_mask > 0] = [1, 0, 0]

original_overlay_trad = np.stack((image_object_trad.image_original,)*3, axis=-1)
original_overlay_trad = normalise_array(original_overlay_trad)
original_overlay_trad[image_object_trad.cleaned_mask > 0] = [1, 0, 0]


highpass_overlay_cellpose = np.stack((image_object_cellpose.high_pass,)*3, axis=-1)
highpass_overlay_cellpose = normalise_array(highpass_overlay_cellpose)
highpass_overlay_cellpose[image_object_cellpose.cleaned_mask > 0] = [1, 0, 0]

original_overlay_cellpose = np.stack((image_object_cellpose.image_original,)*3, axis=-1)
original_overlay_cellpose = normalise_array(original_overlay_cellpose)
original_overlay_cellpose[image_object_cellpose.cleaned_mask > 0] = [1, 0, 0]

# Display data and statistics on the processed image and grains
display(Markdown("## Traditional segmentation"))
fig, ax = plt.subplots(3, 2, figsize=(10, 12))

ax[0,0].imshow(highpass_overlay_trad)
ax[0,0].set_title("Highpassed image overlayed with final mask")
ax[0,0].axis("off")

ax[0,1].imshow(image_object_trad.mask_rgb)
ax[0,1].set_title("Coloured grains with smears removed")
ax[0,1].axis("off")

ax[1,0].imshow(image_object_trad.image_original, cmap="grey")
ax[1,0].set_title("Original Image")
ax[1,0].axis("off")

ax[1,1].imshow(original_overlay_trad)
ax[1,1].set_title("Original image overlayed with final mask")
ax[1,1].axis("off")

sns.histplot(image_object_trad.mask_areas, bins='auto', kde=True, log_scale=True, color='skyblue', edgecolor='black', ax=ax[2,0])
ax[2,0].set_xlabel('Values')
ax[2,0].set_ylabel('Frequency')
ax[2,0].set_title('Grain areas nm²')

sns.histplot(image_object_trad.circularity_data, bins='auto', kde=True, color='skyblue', edgecolor='black', ax=ax[2,1])
ax[2,1].set_xlabel('Values')
ax[2,1].set_ylabel('Frequency')
ax[2,1].set_title('Grain circularities (0-1)')

plt.tight_layout()
plt.show()


display(Markdown("## Cellpose (ML) segmentation"))
fig, ax = plt.subplots(3, 2, figsize=(10, 12))

ax[0,0].imshow(highpass_overlay_cellpose)
ax[0,0].set_title("Highpassed image overlayed with final mask")
ax[0,0].axis("off")

ax[0,1].imshow(image_object_cellpose.mask_rgb)
ax[0,1].set_title("Coloured grains with smears removed")
ax[0,1].axis("off")

ax[1,0].imshow(image_object_cellpose.image_original, cmap="grey")
ax[1,0].set_title("Original Image")
ax[1,0].axis("off")

ax[1,1].imshow(original_overlay_cellpose)
ax[1,1].set_title("Original image overlayed with final mask")
ax[1,1].axis("off")

sns.histplot(image_object_cellpose.mask_areas, bins='auto', kde=True, log_scale=True, color='skyblue', edgecolor='black', ax=ax[2,0])
ax[2,0].set_xlabel('Values')
ax[2,0].set_ylabel('Frequency')
ax[2,0].set_title('Grain areas nm²')

sns.histplot(image_object_cellpose.circularity_data, bins='auto', kde=True, color='skyblue', edgecolor='black', ax=ax[2,1])
ax[2,1].set_xlabel('Values')
ax[2,1].set_ylabel('Frequency')
ax[2,1].set_title('Grain circularities (0-1)')

plt.tight_layout()
plt.show()

---
##### Data (including the config options used in the run) are now exported to the output directory chosen at the top of this notebook:

In [ ]:
logger.info(f"[{image_object_trad.filename}] : *** Exporting data ***")
# Save image and grain data to their own .csv file
image_df = pd.DataFrame([image_object_trad.to_dict()])
grains_list = []
for grain in image_object_trad.grains.values():
    grains_list.append(grain.to_dict())
grain_df = pd.DataFrame(grains_list)

output_filename = f"{output_dir}/{image_object_trad.filename}/image_statistics.csv"
save_to_csv(image_df, output_filename)
output_filename = f"{output_dir}/{image_object_trad.filename}/grain_statistics.csv"
save_to_csv(grain_df, output_filename)
# Save the config settings in a .yaml
output_filename = Path(output_dir) / Path(image_object_trad.filename) / "config.yaml"
save_config(perovstats_object_trad.config, output_filename)

logger.info(
    f"[{image_object.filename}] : Exported data and config to {Path(output_dir) / Path(image_object_trad.filename)}",
)